# Week 2, day 5 (morning) — Worksheet 07: Control flow in a pipeline   (L05)

Where the loops actually go. Every pattern on this sheet is one you will meet
in the first week of any data job: batching, retrying, validating row by row,
joining, processing only what is new, and reporting what happened.

There is nothing new here — it is worksheets 01 to 05 pointed at a job. The
only thing that changes is that a wrong answer now looks like a report rather
than a traceback.

Nothing on this sheet raises. Run the setup cell once, then work down.
**Q11 is the point of the whole sheet.**

Lecture reference: all of `[Python_L05]_Python_control_flow_and_iteration.pdf`.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 07 — Control flow in a pipeline. Run this once.

# 23 record ids waiting to be shipped, for the batching questions.
record_ids = list(range(101, 124))
BATCH_SIZE = 5

# A raw feed, exactly as it arrived: amounts are TEXT, and four rows are bad.
feed = [
    {"id": 101, "ts": 5,  "user": "u1", "amount": "12.50"},
    {"id": 102, "ts": 9,  "user": "u2", "amount": "8.00"},
    {"id": 103, "ts": 14, "user": "u1", "amount": ""},        # blank
    {"id": 104, "ts": 20, "user": "u9", "amount": "31.00"},   # user not on file
    {"id": 105, "ts": 26, "user": "u2", "amount": "-4.00"},   # negative
    {"id": 106, "ts": 31, "user": "u3", "amount": "17.25"},
    {"id": 107, "ts": 38, "user": "u1", "amount": "n/a"},     # not a number
    {"id": 108, "ts": 44, "user": "u3", "amount": "9.99"},
]

# The dimension table the feed is joined to.
customers = {"u1": "north", "u2": "south", "u3": "north"}

# What the delivery endpoint returned, attempt by attempt.
attempt_log = ["timeout", "timeout", "ok"]
flaky_log = ["timeout", "timeout", "timeout", "timeout", "timeout"]

MAX_ATTEMPTS = 5
watermark = 0     # the highest ts already processed on a previous run

print(len(record_ids), "ids,", len(feed), "feed rows,",
      len(customers), "customers on file")

PART A — Batching

### Question 1

Nothing ships one row at a time. Split `record_ids` into batches of `BATCH_SIZE` using `range(0, len(record_ids), BATCH_SIZE)` and a slice. Print each batch with its size, then the number of batches.
> **HINT:** `record_ids[start:start + BATCH_SIZE]` — and a slice that runs off the end of a list does **not** raise. It just stops.

In [ ]:
############################
## Your Code Here
############################

### Question 2

Check the batching adds up. Total the sizes of all the batches and print whether that equals `len(record_ids)`. Then print the number of batches you would get for batch sizes `1`, `5`, `23` and `50`.
> **THE 50 IS THE ONE TO PREDICT.** A batch size bigger than the whole dataset.

In [ ]:
############################
## Your Code Here
############################

PART B — Retrying

### Question 3

The delivery endpoint is unreliable. Walk `attempt_log` inside a `while` loop capped at `MAX_ATTEMPTS`, printing each attempt, and `break` when you see `"ok"`. Print whether it succeeded and how many attempts it took.

In [ ]:
############################
## Your Code Here
############################

### Question 4

Run the same loop against `flaky_log`, which never returns `"ok"`. Print the same two facts afterwards, and write in a comment what this code must **not** do when it gives up.
> **COMPARE THE TWO OUTPUTS.** Only one word differs, and it is the word that matters.

In [ ]:
############################
## Your Code Here
############################

PART C — Validating row by row

### Question 5

Build `clean` and `rejects` from `feed`. Reject a row — with `continue` and a recorded reason — if the amount is blank, if it is not a number, or if it is negative. Rows that survive get their amount converted to a float and added to `clean`. Print how many of each, and print every rejected id with its reason.
> **HINT:** `"12.50".replace(".", "", 1).replace("-", "", 1).isdigit()` is `True`. Work out why before you use it.

In [ ]:
############################
## Your Code Here
############################

### Question 6

Join `clean` to `customers` to get each row's region. Rows whose user is not on file are **orphans**: skip them with `continue` and collect them separately. Print the joined rows, the number of orphans, and the total amount that the orphans took with them.
> **THAT LAST NUMBER IS THE POINT.** An orphan is not a row you lost — it is revenue you lost.

In [ ]:
############################
## Your Code Here
############################

PART D — Only doing the new work

### Question 7

Process only the rows newer than `watermark`: loop `feed`, `continue` past anything with `ts` less than or equal to it, count what you process, and track the highest `ts` you saw. Print the count and the new watermark. Set `watermark = 15` first, so there is something to skip.

In [ ]:
############################
## Your Code Here
############################

### Question 8

Run **exactly the same loop again**, now that `watermark` has moved. Print the same two lines. Then print, in a comment, the difference between this output and the output of a pipeline whose source file failed to arrive.
> **THIS IS THE CORRECT BEHAVIOUR.** It is also indistinguishable from a serious failure.

In [ ]:
############################
## Your Code Here
############################

PART E — What the loop costs

### Question 9

Join `clean` to `customers` twice more, counting the comparisons each way. First with a nested loop over `customers.items()`; then with a dictionary lookup. Print both counts and confirm the two joins agree.
> **FOUR ROWS AGAINST THREE CUSTOMERS.** Scale both by a thousand and read the numbers again.

In [ ]:
############################
## Your Code Here
############################

### Question 10

Report progress every 3 records while walking `feed`, using `enumerate` and `i % 3 == 0`. Print the lines it produces, then say in a comment which record numbers it reported on and whether that is what you would want in a log.

In [ ]:
############################
## Your Code Here
############################

PART F — The run report

### Question 11

Write the summary this pipeline would post to a monitoring channel. First print the version that only says what succeeded: rows delivered and errors raised. Then print the honest version, accounting for **every** row of the original feed. Compare the two.
> **BOTH ARE TRUE.** One of them would let this run pass unnoticed for months.

In [ ]:
############################
## Your Code Here
############################